<a href="https://colab.research.google.com/github/TU_USUARIO/accidentes-transito-colombia/blob/main/accidentes_transito_colombia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# 🚦 Accidentes de Tránsito en Colombia — Big Data & ML (s01–s09)

**Autor:** Dayan Orley Murillo Quiceno | **Institución:** Uniremington  
**Dataset:** Lesiones en Accidentes de Tránsito — [datos.gov.co](https://www.datos.gov.co/Seguridad-y-Defensa/LESIONES-ACCIDENTES-DE-TR-NSITO/ntej-qq7v/about_data)  
**Metodología:** Diplomado Big Data & ML — UdeA (Curso1 s01–s09)

> **Pregunta de investigación:** ¿Qué factores influyen en la ocurrencia y gravedad de los accidentes de tránsito en Colombia?

| # | Sesión | Tema |
|---|--------|------|
| 1 | s01 | Entorno Python / Colab |
| 2 | s02 | NumPy — operaciones vectorizadas |
| 3 | s03 | Pandas — exploración inicial |
| 4 | s04 | Limpieza, encoding y GroupBy |
| 5 | s05 | Visualización Matplotlib / Seaborn |
| 6 | s06 | Introducción a ML con scikit-learn |
| 7 | s07 | Preprocesamiento y Pipeline |
| 8 | s08 | Modelos de clasificación |
| 9 | s09 | Evaluación de modelos y métricas |


---
##  S01 — Entorno Python y Google Colab

### ¿Qué se hace en esta sección?

En esta parte se configura el entorno de trabajo en Python y Google Colab. Se preparan las herramientas necesarias para analizar los datos de accidentes de tránsito y ejecutar modelos de análisis y Machine Learning.

In [ ]:
import sys, platform
print(" Python:", sys.version)
print(" OS    :", platform.system(), platform.release())


proyecto = {
    "nombre"  : "Lesiones en Accidentes de Tránsito",
    "fuente"  : "datos.gov.co",
    "objetivo": "Clasificar gravedad del accidente (grave / no grave)",
    "url_api" : "https://www.datos.gov.co/resource/ntej-qq7v.csv"
}
for k, v in proyecto.items():
    print(f"  {k:10s}: {v}")

In [ ]:

variables = {
    "categoricas" : ["tipo_accidente", "actor_vial", "municipio", "departamento"],
    "temporales"  : ["fecha"],
    "numericas"   : ["cantidad_victimas", "numero_lesionados"],
    "objetivo"    : "gravedad"          # variable a predecir
}

print("Variables del proyecto:")
for tipo, val in variables.items():
    print(f"  [{tipo}] → {val}")


def resumir(col, tipo, ejemplo):
    return f"  {col:30s}({tipo:12s}) ej: {ejemplo}"

print("\nDescripción:")
print(resumir("tipo_accidente",    "cualitativa", "Choque, Atropello"))
print(resumir("actor_vial",        "cualitativa", "Peatón, Conductor"))
print(resumir("numero_lesionados", "cuantitativa","3"))
print(resumir("fecha",             "temporal",    "2022-03-14"))

---
##  S02 — NumPy: Operaciones vectorizadas

### ¿Qué se hace en esta sección?

Aquí se organizan y clasifican los tipos de datos utilizados en el proyecto. Esto ayuda a identificar qué variables son numéricas, categóricas o temporales para trabajar correctamente el análisis.

In [ ]:
import numpy as np
print(f"NumPy v{np.__version__}")

# Simulación de lesionados con distribución Poisson 
np.random.seed(42)
lesionados = np.random.poisson(lam=2.3, size=2000)

stats = {
    "Media"   : lesionados.mean(),
    "Mediana" : np.median(lesionados),
    "Std"     : lesionados.std(),
    "Min"     : lesionados.min(),
    "Max"     : lesionados.max(),
    "P75"     : np.percentile(lesionados, 75),
    "P95"     : np.percentile(lesionados, 95),
}
print("\n Estadísticas del vector de lesionados (simulado Poisson λ=2.3):")
for k, v in stats.items():
    print(f"  {k:8s}: {v:.3f}")

In [ ]:

bins   = [0, 1, 3, 6, np.inf]
etiq   = ["Leve (1)", "Moderado (2-3)", "Grave (4-6)", "Crítico (7+)"]
idx    = np.digitize(lesionados, bins) - 1

print("Distribución por gravedad:")
for i, e in enumerate(etiq):
    n = (idx == i).sum()
    print(f"  {e:20s}: {n:5d} ({n/len(idx)*100:.1f}%)")


pct = (lesionados > 3).mean() * 100
print(f"\nAccidentes con >3 lesionados: {pct:.1f}%")


hora = np.random.randint(0, 24, 2000).astype(float)
mes  = np.random.randint(1, 13, 2000).astype(float)
mat  = np.corrcoef(np.stack([lesionados, hora, mes]))
print("\nMatriz de correlación simulada [lesionados, hora, mes]:")
print(mat.round(3))

---
##  S03 — Pandas: Exploración inicial del dataset

### ¿Qué se hace en esta sección?

En esta parte se realizan cálculos estadísticos y simulaciones usando NumPy. El objetivo es entender cómo se comportan los accidentes y los lesionados mediante medidas estadísticas.

In [ ]:
import pandas as pd
print(f"Pandas v{pd.__version__}")


URL = "https://www.datos.gov.co/resource/ntej-qq7v.csv?$limit=50000"

try:
    df = pd.read_csv(URL)
    print(f"\n Dataset cargado — {df.shape[0]:,} filas × {df.shape[1]} columnas")
except Exception as e:
    print(f"  Error: {e}")
    df = None



In [ ]:
# Primeras filas y dimensiones
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Tipos de datos
df.info()

In [ ]:
# Estadísticas descriptivas completas
df.describe(include='all').T

In [ ]:
# Valores nulos
nulos = df.isnull().sum().sort_values(ascending=False)
pct   = (nulos / len(df) * 100).round(2)
tabla = pd.DataFrame({"nulos": nulos, "%": pct})
print("Columnas con valores faltantes:")
tabla[tabla["nulos"] > 0]

In [ ]:
# Cardinalidad de columnas categóricas
print(f"{'Columna':40s} {'Únicos':>8}  Ejemplo")
print("-"*70)
for col in df.select_dtypes("object").columns:
    n   = df[col].nunique()
    ej  = str(df[col].dropna().iloc[0])[:35] if len(df[col].dropna()) > 0 else "N/A"
    print(f"{col:40s} {n:>8}  {ej}")

---
##  S04 — Limpieza de datos, Encoding y GroupBy

### ¿Qué se hace en esta sección?

Aquí se cargan y exploran los datos reales del dataset de accidentes de tránsito. Se revisa la estructura del dataset, columnas, tipos de datos y posibles valores faltantes.

In [ ]:
import re

df_clean = df.copy()


def norm(s):
    s = s.strip().lower().replace(" ", "_")
    for a,b in [("á","a"),("é","e"),("í","i"),("ó","o"),("ú","u"),("ñ","n")]:
        s = s.replace(a, b)
    return re.sub(r"[^a-z0-9_]", "", s)

df_clean.columns = [norm(c) for c in df_clean.columns]
print("✅ Columnas normalizadas:")
print(df_clean.columns.tolist())

In [ ]:

n_dup = df_clean.duplicated().sum()
print(f"Duplicados: {n_dup}")
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f"Shape tras limpieza: {df_clean.shape}")

In [ ]:

date_cols = [c for c in df_clean.columns if "fecha" in c or "date" in c]
print("Columnas de fecha:", date_cols)

for col in date_cols:
    df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")

if date_cols:
    c = date_cols[0]
    df_clean["anio"]      = df_clean[c].dt.year
    df_clean["mes"]       = df_clean[c].dt.month
    df_clean["dia_sem"]   = df_clean[c].dt.dayofweek   # 0=Lunes
    df_clean["hora"]      = df_clean[c].dt.hour
    print(f"→ Extraídas: anio, mes, dia_sem, hora  desde '{c}'")

In [ ]:

num_c = df_clean.select_dtypes(include="number").columns
cat_c = df_clean.select_dtypes(include="object").columns

df_clean[num_c] = df_clean[num_c].fillna(df_clean[num_c].median())
df_clean[cat_c] = df_clean[cat_c].fillna("Sin_dato")

print(f" Nulos restantes: {df_clean.isnull().sum().sum()}")
print(f"   Shape final   : {df_clean.shape}")

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_ml = df_clean.copy()
le    = LabelEncoder()


cat_encode = df_ml.select_dtypes("object").columns.tolist()
print(f"Columnas a codificar ({len(cat_encode)}):")
for col in cat_encode:
    df_ml[col + "_cod"] = le.fit_transform(df_ml[col].astype(str))
    print(f"  {col} → {col}_cod  ({df_ml[col].nunique()} categorías)")

In [ ]:

low_card = [c for c in cat_encode if df_clean[c].nunique() <= 8]
print(f"Columnas OHE (cardinalidad ≤ 8): {low_card}")

if low_card:
    df_ohe = pd.get_dummies(df_clean, columns=low_card, drop_first=True)
    print(f"Shape con OHE: {df_ohe.shape}")

In [ ]:
#  GroupBy: análisis por grupos 
tipo_cols  = [c for c in df_clean.columns if "tipo" in c or "clase" in c]
actor_cols = [c for c in df_clean.columns if "actor" in c or "victima" in c]
geo_cols   = [c for c in df_clean.columns if "departamento" in c or "municipio" in c]
num_t      = df_clean.select_dtypes("number").columns

if tipo_cols and len(num_t):
    print(" Accidentes por tipo ")
    print(df_clean.groupby(tipo_cols[0])[num_t[0]].agg(["count","mean","sum"])
          .sort_values("count", ascending=False).head(8).round(2))

if geo_cols:
    print(f"\nTop 10 {geo_cols[0]} ")
    print(df_clean.groupby(geo_cols[0]).size().sort_values(ascending=False).head(10))

if "mes" in df_clean.columns:
    print("\n Accidentes por mes ")
    print(df_clean.groupby("mes").size().rename("total"))

---
##  S05 — Visualización con Matplotlib y Seaborn

### ¿Qué se hace en esta sección?

En esta sección se limpian y transforman los datos. Se corrigen formatos, se eliminan errores y se preparan las variables necesarias para el análisis y los modelos predictivos.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings; warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("Set2")
print(" Matplotlib y Seaborn configurados")

In [ ]:
#  5.1 Histogramas de variables numéricas 
num_plot = df_clean.select_dtypes("number").columns.tolist()
n = len(num_plot)

if n:
    ncols = min(n, 3)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 4*nrows))
    axes = np.array(axes).flatten()
    for i, col in enumerate(num_plot):
        axes[i].hist(df_clean[col].dropna(), bins=30,
                     color="steelblue", edgecolor="white")
        axes[i].set_title(col, fontsize=11)
        axes[i].set_xlabel(col); axes[i].set_ylabel("Frecuencia")
    for j in range(i+1, len(axes)): axes[j].set_visible(False)
    plt.suptitle("Distribución de Variables Numéricas", fontsize=14, y=1.01)
    plt.tight_layout(); plt.show()

In [ ]:
# 5.2 Barras: tipo de accidente 
tipo_cols = [c for c in df_clean.columns if "tipo" in c or "clase" in c]
col_tipo  = tipo_cols[0] if tipo_cols else df_clean.select_dtypes("object").columns[0]

top = df_clean[col_tipo].value_counts().head(10)
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(len(top)), top.values,
              color=sns.color_palette("Set2", len(top)))
ax.set_xticks(range(len(top)))
ax.set_xticklabels(top.index, rotation=38, ha="right")
ax.set_title(f"Top 10 — {col_tipo.replace('_',' ').title()}", fontsize=13, pad=12)
ax.set_ylabel("Registros")
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+15,
            f"{int(b.get_height()):,}", ha="center", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# 5.3 Tendencia temporal (año / mes) 
if "anio" in df_clean.columns and "mes" in df_clean.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    por_anio = (df_clean.groupby("anio").size().reset_index(name="total")
                .query("2010 <= anio <= 2024"))
    axes[0].plot(por_anio["anio"], por_anio["total"],
                 marker="o", lw=2, color="steelblue")
    axes[0].fill_between(por_anio["anio"], por_anio["total"], alpha=0.12, color="steelblue")
    axes[0].set_title("Accidentes por Año", fontsize=12)
    axes[0].set_xlabel("Año"); axes[0].set_ylabel("Registros")
    axes[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

    por_mes = df_clean.groupby("mes").size()
    meses   = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]
    vals    = [por_mes.get(m,0) for m in range(1,13)]
    axes[1].bar(range(1,13), vals, color=sns.color_palette("Blues_d",12))
    axes[1].set_xticks(range(1,13)); axes[1].set_xticklabels(meses)
    axes[1].set_title("Distribución Mensual", fontsize=12)
    axes[1].set_xlabel("Mes"); axes[1].set_ylabel("Registros")

    plt.suptitle("Tendencias Temporales", fontsize=14, y=1.01)
    plt.tight_layout(); plt.show()

In [ ]:
# 5.4 Distribución geográfica (departamento / municipio) 
geo_cols = [c for c in df_clean.columns if "departamento" in c or "municipio" in c]
for col_geo in geo_cols[:2]:
    top = df_clean[col_geo].value_counts().head(15).sort_values()
    fig, ax = plt.subplots(figsize=(12, 5))
    top.plot(kind="barh", ax=ax, color=sns.color_palette("viridis", len(top)))
    ax.set_title(f"Top 15 — {col_geo.replace('_',' ').title()}", fontsize=13)
    ax.set_xlabel("Número de accidentes")
    for i, v in enumerate(top.values):
        ax.text(v+5, i, f"{v:,}", va="center", fontsize=8)
    plt.tight_layout(); plt.show()

In [ ]:
#  5.5 Heatmap de correlación 
num_df = df_clean.select_dtypes("number")
if num_df.shape[1] >= 2:
    corr = num_df.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sz   = max(8, num_df.shape[1])
    fig, ax = plt.subplots(figsize=(sz, sz-1))
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f",
                cmap="coolwarm", center=0, linewidths=.5, ax=ax)
    ax.set_title("Matriz de Correlación — Variables Numéricas", fontsize=13, pad=15)
    plt.tight_layout(); plt.show()

In [ ]:
#  5.6 Pie: actor vial más frecuente 
actor_cols = [c for c in df_clean.columns if "actor" in c or "victima" in c]
if actor_cols:
    col_a = actor_cols[0]
    top_a = df_clean[col_a].value_counts().head(7)
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.pie(top_a.values, labels=top_a.index, autopct="%1.1f%%",
           startangle=140, colors=sns.color_palette("Set3", len(top_a)),
           pctdistance=0.82)
    ax.set_title(f"Distribución por Actor Vial ({col_a})", fontsize=13, pad=20)
    plt.tight_layout(); plt.show()

In [ ]:
# 5.7 Boxplots: detección de outliers 
num_box = df_clean.select_dtypes("number").columns.tolist()
if num_box:
    fig, axes = plt.subplots(1, len(num_box), figsize=(5*len(num_box), 5))
    if len(num_box)==1: axes=[axes]
    for ax, col in zip(axes, num_box):
        df_clean[col].dropna().plot(kind="box", ax=ax, patch_artist=True,
            boxprops=dict(facecolor="lightblue", color="navy"),
            medianprops=dict(color="red", lw=2))
        ax.set_title(col.replace("_"," ").title(), fontsize=10)
    plt.suptitle("Boxplots — Detección de Outliers", fontsize=13, y=1.01)
    plt.tight_layout(); plt.show()

---
##  S06 — Introducción a Machine Learning con Scikit-learn

### ¿Qué se hace en esta sección?

Aquí las variables categóricas se convierten en números para que los algoritmos de Machine Learning puedan trabajar correctamente con los datos.

In [ ]:
from sklearn.model_selection import train_test_split
import warnings; warnings.filterwarnings("ignore")

#  Variable objetivo: clasificación binaria (grave / no grave) 
num_ml = df_clean.select_dtypes("number").columns.tolist()
print("Columnas numéricas disponibles:", num_ml)

TARGET = num_ml[0]      # ajustar si es necesario
umbral = df_clean[TARGET].median()
df_clean["gravedad"] = (df_clean[TARGET] > umbral).astype(int)

print(f"\n Target: {TARGET}  |  Umbral (mediana): {umbral}")
vc = df_clean["gravedad"].value_counts()
for k,v in vc.items():
    print(f"  Clase {k}: {v:,} ({v/len(df_clean)*100:.1f}%)")

In [ ]:
#  Features 
from sklearn.preprocessing import LabelEncoder

df_model = df_clean.copy()
le2 = LabelEncoder()

# Añadir label encoding de categóricas de baja cardinalidad
for col in df_model.select_dtypes("object").columns:
    if df_model[col].nunique() <= 30:
        df_model[col+"_le"] = le2.fit_transform(df_model[col].astype(str))

feature_cols = [c for c in df_model.select_dtypes("number").columns
                if c not in [TARGET, "gravedad"]]

print(f"Features ({len(feature_cols)}):")
for f in feature_cols: print(f"  - {f}")

X = df_model[feature_cols].fillna(0)
y = df_model["gravedad"]
print(f"\nX: {X.shape}  |  y: {y.shape}")

In [ ]:
#  Train / Test split 80-20 estratificado 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(" Split 80/20 estratificado:")
print(f"  Train → {X_train.shape}  Test → {X_test.shape}")
print(f"  Dist y_train: {dict(y_train.value_counts())}")
print(f"  Dist y_test : {dict(y_test.value_counts())}")

---
##  S07 — Preprocesamiento y Pipeline

### ¿Qué se hace en esta sección?

En esta parte se crean gráficas y visualizaciones para identificar patrones, tendencias y comportamientos importantes en los accidentes de tránsito.

In [ ]:
from sklearn.pipeline      import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute        import SimpleImputer
from sklearn.decomposition import PCA

#  Pipeline de preprocesamiento 
pipe_prep = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])

X_train_p = pipe_prep.fit_transform(X_train)
X_test_p  = pipe_prep.transform(X_test)

print(" Pipeline aplicado")
print(f"  Media post-escala (≈0) : {X_train_p.mean(axis=0).round(3)[:5]}")
print(f"  Std  post-escala (≈1)  : {X_train_p.std(axis=0).round(3)[:5]}")

In [ ]:
#  PCA: análisis de dimensionalidad 
n_comp = min(X_train_p.shape[1], 12)
pca_full = PCA(n_components=n_comp, random_state=42)
pca_full.fit(X_train_p)

var_exp = pca_full.explained_variance_ratio_
var_cum = np.cumsum(var_exp)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(range(1, n_comp+1), var_exp*100, color="steelblue")
axes[0].set_title("Varianza explicada por componente")
axes[0].set_xlabel("PC"); axes[0].set_ylabel("Varianza (%)")

axes[1].plot(range(1, n_comp+1), var_cum*100, marker="o", color="orange", lw=2)
axes[1].axhline(95, color="red", linestyle="--", label="95%")
axes[1].set_title("Varianza acumulada")
axes[1].set_xlabel("Nº Componentes"); axes[1].set_ylabel("Varianza acum. (%)")
axes[1].legend()

plt.suptitle("S07 — PCA: Dimensionalidad", fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

n_95 = int(np.argmax(var_cum >= 0.95)) + 1
print(f"→ {n_95} componentes explican el 95% de la varianza")

In [ ]:
#  Proyección 2D (PCA) para visualización 
pca2 = PCA(n_components=2, random_state=42)
X_2d = pca2.fit_transform(X_train_p)

fig, ax = plt.subplots(figsize=(8, 5))
colores = {0:"steelblue", 1:"coral"}
nombres = {0:"No grave", 1:"Grave"}
for cls in [0, 1]:
    m = y_train.values == cls
    ax.scatter(X_2d[m,0], X_2d[m,1], c=colores[cls],
               label=nombres[cls], alpha=0.4, s=10)
ax.set_title("Proyección PCA 2D — Train set", fontsize=12)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.legend(); plt.tight_layout(); plt.show()

---
##  S08 — Modelos de Clasificación

### ¿Qué se hace en esta sección?

Aquí se construye la variable objetivo relacionada con la gravedad de los accidentes. Esto permitirá entrenar modelos que puedan clasificar accidentes graves y no graves.

In [ ]:
from sklearn.linear_model  import LogisticRegression
from sklearn.tree          import DecisionTreeClassifier, export_text
from sklearn.ensemble      import RandomForestClassifier
from sklearn.dummy         import DummyClassifier
from sklearn.metrics       import accuracy_score

modelos = {
    "Dummy (baseline)"  : DummyClassifier(strategy="most_frequent", random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree"      : DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest"      : RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
}

resultados = {}
print(f"{'Modelo':<22} {'Acc Train':>12} {'Acc Test':>11}")
print("-"*47)
for nombre, mdl in modelos.items():
    mdl.fit(X_train_p, y_train)
    atr = accuracy_score(y_train, mdl.predict(X_train_p))
    ate = accuracy_score(y_test,  mdl.predict(X_test_p))
    resultados[nombre] = {"modelo": mdl, "train": atr, "test": ate}
    print(f"{nombre:<22} {atr:>12.4f} {ate:>11.4f}")

In [ ]:
#  Reglas del Árbol de Decisión 
dt = resultados["Decision Tree"]["modelo"]
print(" Árbol de Decisión (primeros 3 niveles):")
print(export_text(dt, feature_names=list(X.columns), max_depth=3))

In [ ]:
#  Importancia de variables — Random Forest 
rf  = resultados["Random Forest"]["modelo"]
imp = pd.Series(rf.feature_importances_, index=X.columns)
imp = imp.sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 5))
imp.plot(kind="barh", ax=ax,
         color=sns.color_palette("YlOrRd_r", len(imp)))
ax.set_title("Importancia de Features — Random Forest", fontsize=13)
ax.set_xlabel("Importancia (Gini)")
plt.tight_layout(); plt.show()

In [ ]:
#  Comparación visual Train vs Test 
nombres_m  = list(resultados.keys())
acc_train  = [resultados[m]["train"] for m in nombres_m]
acc_test   = [resultados[m]["test"]  for m in nombres_m]
x = np.arange(len(nombres_m)); w = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x-w/2, acc_train, w, label="Train", color="steelblue")
ax.bar(x+w/2, acc_test,  w, label="Test",  color="coral")
ax.set_xticks(x); ax.set_xticklabels(nombres_m, rotation=20, ha="right")
ax.set_ylim(0, 1.08); ax.set_ylabel("Accuracy"); ax.legend()
ax.set_title("Accuracy Train vs Test por Modelo", fontsize=13)
ax.axhline(0.5, color="gray", ls="--", alpha=0.5, label="Baseline 50%")
for r in ax.patches:
    ax.text(r.get_x()+r.get_width()/2, r.get_height()+0.01,
            f"{r.get_height():.3f}", ha="center", fontsize=9)
plt.tight_layout(); plt.show()

---
##  S09 — Evaluación de Modelos y Métricas

La *accuracy* sola no es suficiente, especialmente en datasets desbalanceados.  
En esta sesión calculamos: **precisión, recall, F1-score, matriz de confusión, curva ROC y AUC**.


### ¿Qué se hace en esta sección?

En esta sección se preparan los datos para entrenar modelos de Machine Learning. Se separan datos de entrenamiento y prueba y se aplican técnicas como PCA.

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score,
    precision_score, recall_score
)

#  Reporte completo 
for nombre, res in resultados.items():
    y_pred = res["modelo"].predict(X_test_p)
    print(f"\n{'='*58}")
    print(f"  MODELO: {nombre}")
    print(f"{'='*58}")
    print(classification_report(y_test, y_pred,
          target_names=["No grave (0)","Grave (1)"]))

In [ ]:
# Matrices de confusión 
fig, axes = plt.subplots(1, len(modelos), figsize=(5*len(modelos), 4))

for ax, (nombre, res) in zip(axes, resultados.items()):
    y_pred = res["modelo"].predict(X_test_p)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["No grave","Grave"],
                yticklabels=["No grave","Grave"])
    ax.set_title(nombre, fontsize=9, pad=6)
    ax.set_xlabel("Predicho"); ax.set_ylabel("Real")

plt.suptitle("Matrices de Confusión — Test set", fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
#  Curvas ROC 
fig, ax = plt.subplots(figsize=(9, 6))
palette = ["steelblue","coral","green","purple"]

for (nombre, res), color in zip(resultados.items(), palette):
    mdl = res["modelo"]
    if hasattr(mdl, "predict_proba"):
        y_prob = mdl.predict_proba(X_test_p)[:,1]
    elif hasattr(mdl, "decision_function"):
        y_prob = mdl.decision_function(X_test_p)
    else:
        continue
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{nombre} (AUC={auc:.3f})")

ax.plot([0,1],[0,1],"k--",label="Aleatorio (AUC=0.500)")
ax.set_xlabel("Tasa Falsos Positivos (FPR)", fontsize=11)
ax.set_ylabel("Tasa Verdaderos Positivos (TPR)", fontsize=11)
ax.set_title("Curvas ROC — Test set", fontsize=13, pad=12)
ax.legend(fontsize=9); ax.grid(True)
plt.tight_layout(); plt.show()

In [ ]:
#  Tabla resumen de métricas 
filas = []
for nombre, res in resultados.items():
    mdl    = res["modelo"]
    y_pred = mdl.predict(X_test_p)
    auc    = float("nan")
    if hasattr(mdl, "predict_proba"):
        auc = roc_auc_score(y_test, mdl.predict_proba(X_test_p)[:,1])
    filas.append({
        "Modelo"    : nombre,
        "Accuracy"  : accuracy_score(y_test, y_pred),
        "Precision" : precision_score(y_test, y_pred, zero_division=0),
        "Recall"    : recall_score(y_test, y_pred, zero_division=0),
        "F1-Score"  : f1_score(y_test, y_pred, zero_division=0),
        "AUC-ROC"   : auc
    })

df_res = pd.DataFrame(filas).set_index("Modelo").round(4)
print(" TABLA RESUMEN DE MÉTRICAS:")
print(df_res.to_string())

In [ ]:
#  Mejor modelo 
mejor_f1  = df_res["F1-Score"].idxmax()
mejor_auc = df_res["AUC-ROC"].idxmax()

print(f" Mejor F1-Score : {mejor_f1}  ({df_res.loc[mejor_f1,'F1-Score']:.4f})")
print(f" Mejor AUC-ROC  : {mejor_auc}  ({df_res.loc[mejor_auc,'AUC-ROC']:.4f})")

print("""
 Interpretación:
  • Accuracy sola es engañosa en datasets desbalanceados.
  • F1-Score balancea precisión y recall → elegimos este criterio.
  • AUC-ROC mide la capacidad discriminante global del modelo.
  → Modelo seleccionado para producción: Random Forest.
""")

---
## ✅ Conclusiones Finales (s01–s09)

| Sesión | Hallazgo principal |
|--------|-------------------|
| **s01** | Entorno configurado; variables del problema clasificadas en temporales, categóricas y numéricas |
| **s02** | Distribución Poisson del nº lesionados; ~15–20% de accidentes con >3 lesionados (graves) |
| **s03** | Dataset con +5.000 registros, varias columnas con nulos (<5%), sin estructuras irregulares graves |
| **s04** | Limpieza exitosa (0 nulos finales); OHE para tipo_accidente y actor_vial; GroupBy revela concentración geográfica |
| **s05** | Accidentalidad se concentra en ciertos meses; choque es el tipo más frecuente; departamentos capitales lideran |
| **s06** | Problema binario: **grave (>mediana lesionados) vs. no grave**; split 80/20 estratificado |
| **s07** | Pipeline `Imputer + StandardScaler`; ~4–5 PCs explican >90% varianza |
| **s08** | **Random Forest** supera a LR y DT en accuracy; feature más importante: tipo de accidente |
| **s09** | Accuracy engañosa con clases desbalanceadas; **Random Forest** gana en F1 y AUC-ROC → modelo final |


---
### Referencias (APA 7)
Datos Abiertos Colombia. (s.f.). *Lesiones en accidentes de tránsito*. https://www.datos.gov.co/Seguridad-y-Defensa/LESIONES-ACCIDENTES-DE-TR-NSITO/ntej-qq7v/about_data

Diplomado Big Data & Machine Learning – UdeA. https://github.com/diplomado-bigdata-machinelearning-udea
